Here is a project to detect "לשון הרע "

In [31]:
#%pip install pandas

Here we load the dataset
The dataset is available at https://www.kaggle.com/kazanova/sentiment-analysis-in-texts

In [30]:

import pandas as pd

df = pd.read_csv('./training.1600000.processed.noemoticon.csv', encoding='ISO-8859-1')
df.head()


,0,1467810369,Mon Apr 06 22:19:45 PDT 2009,NO_QUERY,_TheSpecialOne_,"@switchfoot http://twitpic.com/2y1zl - Awww, that's a bummer. You shoulda got David Carr of Third Day to do it. ;D"
0,0,1467810672,Mon Apr 06 22:19:49 PDT 2009,NO_QUERY,scotthamilton,is upset that he can't update his Facebook by ...
1,0,1467810917,Mon Apr 06 22:19:53 PDT 2009,NO_QUERY,mattycus,@Kenichan I dived many times for the ball. Man...
2,0,1467811184,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,ElleCTF,my whole body feels itchy and like its on fire
3,0,1467811193,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,Karoli,"@nationwideclass no, it's not behaving at all...."
4,0,1467811372,Mon Apr 06 22:20:00 PDT 2009,NO_QUERY,joy_wolf,@Kwesidei not the whole crew


Here we will add columns names:

In [31]:
df.columns = ['sentiment', 'id', 'date', 'query', 'user', 'text']
df.head()

,sentiment,id,date,query,user,text
0,0,1467810672,Mon Apr 06 22:19:49 PDT 2009,NO_QUERY,scotthamilton,is upset that he can't update his Facebook by ...
1,0,1467810917,Mon Apr 06 22:19:53 PDT 2009,NO_QUERY,mattycus,@Kenichan I dived many times for the ball. Man...
2,0,1467811184,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,ElleCTF,my whole body feels itchy and like its on fire
3,0,1467811193,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,Karoli,"@nationwideclass no, it's not behaving at all...."
4,0,1467811372,Mon Apr 06 22:20:00 PDT 2009,NO_QUERY,joy_wolf,@Kwesidei not the whole crew


Why do we need 'query' column?

In [32]:
# Here we will check the different values in the 'query' column:
print(df['query'].unique())

['NO_QUERY']


It only has one value, so we can drop this column, 
but first we will copy the dateset:

In [33]:
data = df.copy()
data = data.drop(columns=['query'])
data.head()

,sentiment,id,date,user,text
0,0,1467810672,Mon Apr 06 22:19:49 PDT 2009,scotthamilton,is upset that he can't update his Facebook by ...
1,0,1467810917,Mon Apr 06 22:19:53 PDT 2009,mattycus,@Kenichan I dived many times for the ball. Man...
2,0,1467811184,Mon Apr 06 22:19:57 PDT 2009,ElleCTF,my whole body feels itchy and like its on fire
3,0,1467811193,Mon Apr 06 22:19:57 PDT 2009,Karoli,"@nationwideclass no, it's not behaving at all...."
4,0,1467811372,Mon Apr 06 22:20:00 PDT 2009,joy_wolf,@Kwesidei not the whole crew


Same for the id column, we don't think it can help to tell if a given text is לשון הרע or not:

In [34]:
data = data.drop(columns=['id'])
data.head()

,sentiment,date,user,text
0,0,Mon Apr 06 22:19:49 PDT 2009,scotthamilton,is upset that he can't update his Facebook by ...
1,0,Mon Apr 06 22:19:53 PDT 2009,mattycus,@Kenichan I dived many times for the ball. Man...
2,0,Mon Apr 06 22:19:57 PDT 2009,ElleCTF,my whole body feels itchy and like its on fire
3,0,Mon Apr 06 22:19:57 PDT 2009,Karoli,"@nationwideclass no, it's not behaving at all...."
4,0,Mon Apr 06 22:20:00 PDT 2009,joy_wolf,@Kwesidei not the whole crew


Here we will check the amount of rows of each type: 
0 - negative text
1 - positive text 

In [35]:
print(data['sentiment'].value_counts().loc[[0, 4]])

sentiment
0    799999
4    800000
Name: count, dtype: int64


In [36]:
print("Example with sentiment 0:")
print(data[data['sentiment'] == 0].iloc[0])

print("\nExample with sentiment 4:")
print(data[data['sentiment'] == 4].iloc[0])

Example with sentiment 0:
sentiment                                                    0
date                              Mon Apr 06 22:19:49 PDT 2009
user                                             scotthamilton
text         is upset that he can't update his Facebook by ...
Name: 0, dtype: object

Example with sentiment 4:
sentiment                                               4
date                         Mon Apr 06 22:22:45 PDT 2009
user                                                ersle
text         I LOVE @Health4UandPets u guys r the best!! 
Name: 799999, dtype: object


to do - Check if rows with label are affected by the time? 
is text written late at night probably labeled negative ?  

We assume that rows labeled 0 that mentions someone else, are more likely to be לשון הרע  for example: 
 0 - "@nationwideclass no, it's not behaving at all...."  - לשון הרע 
 0 - "my whole body feels itchy and like its on fire" - לא לשון הרע 

So here we go and filter all rows that are negative sentiment (labeled 0) and contains '@'

In [37]:
neg_with_mentions = data[(data['sentiment'] == 0) & (data['text'].str.contains('@'))]
print("Number of negative tweets with mentions:", len(neg_with_mentions))

Number of negative tweets with mentions: 305210


About half from the data set with labeled 0

now we can get text like this: "@Kwesidei not the whole crew" - its negative but not לשון הרע 

So now we will find the text that are not only negative but also insulting or offensive.
We do that by creating a list a not good words that if where in a sentence and mention some one we will say the sentence will be לשון הרע

In [38]:
keywords = [
    'stupid', 
    'hate',
    'hates',
    'hate this',
    'hates that',
    'cry',
    'idiot',
    'idiots',
    'worst', 
    'annoying', 
    'ugly',
    'terrible',
    'awful',
    'horrible',
    'horrid',
    'dislike',
    'disgusting',
    'disgusted',
    'disappointed',
    'liar', 
    'bad', 
    'dumb',
    'fool',
    'foolish', 
    'not good',
    'not great',
    'not ok',   
    'not okay',
    'not nice',
    'not like',
    'not happy',
    'upset',
    'iam angry',
    'angry',
    ]

In [39]:
def is_weak_lashon(text):
    text = text.lower()
    return any(word in text for word in keywords)

data['lashon_hara_guess'] = data.apply(
    lambda row: int(row['sentiment'] == 4 and '@' in row['text'] and is_weak_lashon(row['text'])),
    axis=1
)

Lets see the dataset now: 

In [42]:
data.head()

,sentiment,date,user,text,lashon_hara_guess
0,0,Mon Apr 06 22:19:49 PDT 2009,scotthamilton,is upset that he can't update his Facebook by ...,0
1,0,Mon Apr 06 22:19:53 PDT 2009,mattycus,@Kenichan I dived many times for the ball. Man...,0
2,0,Mon Apr 06 22:19:57 PDT 2009,ElleCTF,my whole body feels itchy and like its on fire,0
3,0,Mon Apr 06 22:19:57 PDT 2009,Karoli,"@nationwideclass no, it's not behaving at all....",0
4,0,Mon Apr 06 22:20:00 PDT 2009,joy_wolf,@Kwesidei not the whole crew,0


Lets make sure that the data set are balanced:

In [43]:
data['lashon_hara_guess'].value_counts()
# 1 = likely Lashon Hara, 0 = unlikely

lashon_hara_guess
0    1587844
1      12155
Name: count, dtype: int64

Here we have more rows with labels 0. 
To have the same amount of data for 0 and for 1

In [44]:
# Separate the two classes
class_0 = data[data['lashon_hara_guess'] == 0]
class_1 = data[data['lashon_hara_guess'] == 1]

# Downsample the majority class
min_count = min(len(class_0), len(class_1))
class_0_balanced = class_0.sample(n=min_count, random_state=42)
class_1_balanced = class_1.sample(n=min_count, random_state=42)

# Combine and shuffle
balanced_data = pd.concat([class_0_balanced, class_1_balanced]).sample(frac=1, random_state=42).reset_index(drop=True)
print(balanced_data['lashon_hara_guess'].value_counts())

lashon_hara_guess
0    12155
1    12155
Name: count, dtype: int64


In [68]:
print("Example with sentiment 1 - likely:")
balanced_data[balanced_data['lashon_hara_guess'] == 1].sample()[['text']]['text'].values[0]


Example with sentiment 1 - likely:


"@kateisbored I can hope it didn't happen...it was AWFUL....maybe my mind is playing tricks on me "

In [57]:
print("Example with sentiment 0 - unlikely:")
balanced_data[balanced_data['lashon_hara_guess'] == 0].sample()[['text']]['text'].values[0]

Example with sentiment 0 - unlikely:


'@ahier The ULTIMATE in #FollowFriday for #Healthcare @DrJennifer no one could keep up  THX!'

Problem: this data is not really לשון הרע 
or filter it better or find a different data set !!!!!!!!!!!!!!!!!!!!

So were going to try a few methods: 
- ML
- Deep learning 
- Transformers 

then compare the results:

--  Machine Learning -- 
We used a logistic Regression classifier
Why It's Good:
- Fast to train
- Interpretable (you can inspect weights)
- Surprisingly effective for text tasks

In [46]:
# %pip install scikit-learn

In [47]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

# 1. Load your existing DataFrame
# (assuming df already has 'text' and 'lashon_hara_guess' columns)

# 2. Prepare features and labels
X = balanced_data['text']
y = balanced_data['lashon_hara_guess']

# 3. Split into train and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 4. Convert text to TF-IDF features
vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1,2), stop_words='english')
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

# 5. Train a Logistic Regression model
model = LogisticRegression()
model.fit(X_train_vec, y_train)

# 6. Predict on the test set
y_pred = model.predict(X_test_vec)

# 7. Evaluate
print(classification_report(y_test, y_pred, target_names=["Not Lashon Hara", "Lashon Hara"]))


                 precision    recall  f1-score   support

Not Lashon Hara       0.81      0.88      0.84      2391
    Lashon Hara       0.87      0.79      0.83      2390

       accuracy                           0.84      4781
      macro avg       0.84      0.84      0.84      4781
   weighted avg       0.84      0.84      0.84      4781



In [48]:
def predict_lashon_hara(text):
    vec = vectorizer.transform([text])
    pred = model.predict(vec)[0]
    return "Lashon Hara" if pred == 1 else "Not Lashon Hara"

# Try it!
print(predict_lashon_hara("@john you're such an idiot"))


Lashon Hara


-- Deep learning --
Recurrent Neural Network (RNN) – Text classifier using Keras (TensorFlow) with an embedding layer, LSTM, and Dense layers
It's more powerful than models like logistic regression or simple feedforward networks 
because it can learn temporal dependencies, like:
Text:
“@david wow you're just the smartest guy in the room 🙄”

LSTM will think:
It captures the sarcasm from context and emoji.
Logistic regression: 
might be fooled by the word "smartest".

In [ ]:
# %pip install tensorflow


In [50]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from tensorflow.keras.utils import to_categorical

# 1. Prepare Data
X = balanced_data['text'].values
y = balanced_data['lashon_hara_guess'].values

# 2. Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# 3. Tokenize & Pad
vocab_size = 10000
maxlen = 50

tokenizer = Tokenizer(num_words=vocab_size, oov_token='<OOV>')
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

X_train_pad = pad_sequences(X_train_seq, maxlen=maxlen, padding='post', truncating='post')
X_test_pad = pad_sequences(X_test_seq, maxlen=maxlen, padding='post', truncating='post')

# 4. Model
model = Sequential([
    Embedding(input_dim=vocab_size, output_dim=128, input_length=maxlen),
    LSTM(64, return_sequences=False),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dropout(0.2),
    Dense(1, activation='sigmoid')  # Binary classification
])

model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

# 5. Train
model.fit(X_train_pad, y_train, epochs=5, batch_size=64, validation_split=0.1)

# 6. Evaluate
loss, acc = model.evaluate(X_test_pad, y_test)
print(f"Test Accuracy: {acc:.4f}")


Epoch 1/5


C:\Users\shmue\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\keras\src\layers\core\embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


269/269 ━━━━━━━━━━━━━━━━━━━━ 13s 37ms/step - accuracy: 0.5013 - loss: 0.6937 - val_accuracy: 0.5274 - val_loss: 0.6930
Epoch 2/5
269/269 ━━━━━━━━━━━━━━━━━━━━ 9s 34ms/step - accuracy: 0.5461 - loss: 0.6790 - val_accuracy: 0.7423 - val_loss: 0.5607
Epoch 3/5
269/269 ━━━━━━━━━━━━━━━━━━━━ 10s 36ms/step - accuracy: 0.7147 - loss: 0.5673 - val_accuracy: 0.8390 - val_loss: 0.3786
Epoch 4/5
269/269 ━━━━━━━━━━━━━━━━━━━━ 9s 35ms/step - accuracy: 0.8870 - loss: 0.3202 - val_accuracy: 0.8876 - val_loss: 0.3118
Epoch 5/5
269/269 ━━━━━━━━━━━━━━━━━━━━ 9s 34ms/step - accuracy: 0.9325 - loss: 0.2125 - val_accuracy: 0.8955 - val_loss: 0.2866
150/150 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.8914 - loss: 0.3029
Test Accuracy: 0.8866


Accuracy: 0.8866  

In [ ]:
# Example 1:
print("@dude you're such a liar and an idiot")
print(predict_lashon_hara("@dude you're such a liar and an idiot"))

print("\n")
# Example 2:
print("@dude you're such a liar")
print(predict_lashon_hara("@dude you're such a liar"))

@dude you're such a liar and an idiot
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 60ms/step
('Lashon Hara', 0.7197285890579224)


@dude you're such a liar
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step
('Not Lashon Hara', 0.26234349608421326)


-- DM modal: Embedding + Global Average Pooling + Dense (MLP) -- 
GlobalAveragePooling1D - Averages all word embeddings into a single vector
Suppose you have a sentence of 5 words, and each word is represented by a 4-dimensional embedding:

[
 [0.1, 0.2, 0.3, 0.4],  ← word 1
 [0.5, 0.1, 0.0, 0.3],  ← word 2
 [0.2, 0.3, 0.5, 0.2],  ← word 3
 [0.0, 0.1, 0.2, 0.1],  ← word 4
 [0.1, 0.2, 0.2, 0.3]   ← word 5
]
GlobalAveragePooling1D takes the average across all rows, like this:

mean of each column = [
 (0.1+0.5+0.2+0.0+0.1)/5 = 0.18,
 (0.2+0.1+0.3+0.1+0.2)/5 = 0.18,
 (0.3+0.0+0.5+0.2+0.2)/5 = 0.24,
 (0.4+0.3+0.2+0.1+0.3)/5 = 0.26
]
So now your sentence becomes just one vector of 4 numbers, not 5 vectors of 4:
→ [0.18, 0.18, 0.24, 0.26]

advantages: 
- Much faster than LSTM
- Captures general topic/word presence but not word order
- Good for short, simple sentences or when you don’t need context modeling

| Feature              | This Model         | LSTM                         |
| -------------------- | ------------------ | ---------------------------  |
| Captures word order? | ❌ No              | ✅ Yes                      |
| Training speed       | ⚡ Fast            | 🐢 Slower                   |
| Accuracy potential   | 😐 Medium          | 🔥 Higher (on complex data) |
| Complexity           | 🟢 Simple          | 🔴 More complex             |



In [71]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, GlobalAveragePooling1D, Dense, Dropout

# 1. Prepare Data
X = balanced_data['text'].values
y = balanced_data['lashon_hara_guess'].values

# 2. Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# 3. Tokenize & Pad
vocab_size = 10000
maxlen = 50

tokenizer = Tokenizer(num_words=vocab_size, oov_token='<OOV>')
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

X_train_pad = pad_sequences(X_train_seq, maxlen=maxlen, padding='post', truncating='post')
X_test_pad = pad_sequences(X_test_seq, maxlen=maxlen, padding='post', truncating='post')

# 4. Model

model = Sequential([
    Embedding(input_dim=vocab_size, output_dim=128, input_length=maxlen),
    GlobalAveragePooling1D(),  # turns 2D sequences into flat vector
    Dense(128, activation='relu'),
    Dropout(0.3),
    Dense(64, activation='relu'),
    Dropout(0.2),
    Dense(1, activation='sigmoid')
])


model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# 5. Train
model.fit(X_train_pad, y_train, epochs=5, batch_size=64, validation_split=0.1)

# 6. Evaluate
loss, acc = model.evaluate(X_test_pad, y_test)
print(f"Test Accuracy: {acc:.4f}")

Epoch 1/5


C:\Users\shmue\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\keras\src\layers\core\embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


269/269 ━━━━━━━━━━━━━━━━━━━━ 11s 23ms/step - accuracy: 0.6078 - loss: 0.6378 - val_accuracy: 0.7710 - val_loss: 0.4708
Epoch 2/5
269/269 ━━━━━━━━━━━━━━━━━━━━ 5s 18ms/step - accuracy: 0.8352 - loss: 0.3770 - val_accuracy: 0.8693 - val_loss: 0.3143
Epoch 3/5
269/269 ━━━━━━━━━━━━━━━━━━━━ 5s 19ms/step - accuracy: 0.8976 - loss: 0.2730 - val_accuracy: 0.8766 - val_loss: 0.2924
Epoch 4/5
269/269 ━━━━━━━━━━━━━━━━━━━━ 5s 17ms/step - accuracy: 0.9187 - loss: 0.2179 - val_accuracy: 0.8348 - val_loss: 0.4411
Epoch 5/5
269/269 ━━━━━━━━━━━━━━━━━━━━ 5s 17ms/step - accuracy: 0.9243 - loss: 0.2100 - val_accuracy: 0.8866 - val_loss: 0.3013
150/150 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8697 - loss: 0.3355
Test Accuracy: 0.8728


-- DL: Multilayer Perceptron (MLP) for Text Classification --
Model Type:
A fully connected feedforward neural network
No awareness of word order or sequence, just uses numerical input vectors

Uses TfidfVectorizer - Converts text into a flat vector of word/token importance (no word order)

In [73]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split

# 1. Prepare Data
X = balanced_data['text'].values
y = balanced_data['lashon_hara_guess'].values

# 2. Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

# 3. TF-IDF
vectorizer = TfidfVectorizer(max_features=5000)
X_train_vec = vectorizer.fit_transform(X_train).toarray()
X_test_vec = vectorizer.transform(X_test).toarray()

# 4. MLP Model
model = Sequential([
    Dense(128, activation='relu', input_shape=(X_train_vec.shape[1],)),
    Dropout(0.3),
    Dense(64, activation='relu'),
    Dropout(0.2),
    Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# 5. Train
model.fit(X_train_vec, y_train, epochs=5, batch_size=64, validation_split=0.1)

# 6. Evaluate
loss, acc = model.evaluate(X_test_vec, y_test)
print(f"Test Accuracy: {acc:.4f}")


C:\Users\shmue\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\keras\src\layers\core\dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/5
269/269 ━━━━━━━━━━━━━━━━━━━━ 6s 12ms/step - accuracy: 0.6611 - loss: 0.5883 - val_accuracy: 0.8557 - val_loss: 0.3374
Epoch 2/5
269/269 ━━━━━━━━━━━━━━━━━━━━ 3s 11ms/step - accuracy: 0.8968 - loss: 0.2750 - val_accuracy: 0.8719 - val_loss: 0.3279
Epoch 3/5
269/269 ━━━━━━━━━━━━━━━━━━━━ 3s 11ms/step - accuracy: 0.9324 - loss: 0.1999 - val_accuracy: 0.8657 - val_loss: 0.3463
Epoch 4/5
269/269 ━━━━━━━━━━━━━━━━━━━━ 3s 11ms/step - accuracy: 0.9490 - loss: 0.1512 - val_accuracy: 0.8615 - val_loss: 0.3932
Epoch 5/5
269/269 ━━━━━━━━━━━━━━━━━━━━ 3s 11ms/step - accuracy: 0.9661 - loss: 0.1077 - val_accuracy: 0.8484 - val_loss: 0.4578
150/150 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8455 - loss: 0.4607
Test Accuracy: 0.8417


-- DL: CNN --

A 1D CNN designed to detect local patterns in sequences of word embeddings for text classification

Summary Table

| Feature                   | CNN for Text  |
| ------------------------- | ------------  |
| Learns word order?        | ✅ (local)    |
| Good for sarcasm/context? | ❌ Limited    |
| Great for phrase patterns | ✅ Yes        |
| Requires padding & tokens | ✅ Yes        |
| Training speed            | ⚡ Fast       |



In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Conv1D, GlobalMaxPooling1D, Dense

model = Sequential([
    Embedding(input_dim=vocab_size, output_dim=128, input_length=maxlen),
    Conv1D(128, kernel_size=5, activation='relu'),
    GlobalMaxPooling1D(),
    Dense(64, activation='relu'),
    Dense(1, activation='sigmoid')
])
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

model.fit(X_train_pad, y_train, epochs=5, batch_size=64, validation_split=0.1)
             
loss, acc = model.evaluate(X_test_pad, y_test)
print(f"Test Accuracy: {acc:.4f}")

Epoch 1/5


C:\Users\shmue\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\keras\src\layers\core\embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


269/269 ━━━━━━━━━━━━━━━━━━━━ 10s 27ms/step - accuracy: 0.7399 - loss: 0.5115 - val_accuracy: 0.9064 - val_loss: 0.2622
Epoch 2/5
269/269 ━━━━━━━━━━━━━━━━━━━━ 7s 25ms/step - accuracy: 0.9288 - loss: 0.2037 - val_accuracy: 0.9164 - val_loss: 0.2473
Epoch 3/5
269/269 ━━━━━━━━━━━━━━━━━━━━ 10s 24ms/step - accuracy: 0.9706 - loss: 0.0920 - val_accuracy: 0.8965 - val_loss: 0.3060
Epoch 4/5
269/269 ━━━━━━━━━━━━━━━━━━━━ 7s 24ms/step - accuracy: 0.9895 - loss: 0.0399 - val_accuracy: 0.8897 - val_loss: 0.4135
Epoch 5/5
269/269 ━━━━━━━━━━━━━━━━━━━━ 6s 24ms/step - accuracy: 0.9954 - loss: 0.0174 - val_accuracy: 0.8939 - val_loss: 0.4610
150/150 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.8870 - loss: 0.4781
Test Accuracy: 0.8902


Accuracy: 0.8902

-- Bidirectional LSTM (BiLSTM) for Text Classification ---
Model Type:
- A Recurrent Neural Network (RNN) using LSTM cells
- Bidirectional = it reads the sequence left-to-right and right-to-left
- Great for capturing long-range and contextual meaning

| Feature                          | Value                        |
| -------------------------------- | ---------------------------  |
| Learns word order?               | ✅ Yes                      |
| Captures context from both ends? | ✅✅ Yes (bidirectional)    |
| Handles long sentences?          | ✅ Yes                      |
| Training time?                   | 🟡 Medium (slower than CNN) |


In [75]:
from tensorflow.keras.layers import Bidirectional, LSTM

model = Sequential([
    Embedding(input_dim=vocab_size, output_dim=128, input_length=maxlen),
    Bidirectional(LSTM(64)),
    Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
# 5. Train 

model.fit(X_train_pad, y_train, epochs=5, batch_size=64, validation_split=0.1)
# 6. Evaluate               
loss, acc = model.evaluate(X_test_pad, y_test)
print(f"Test Accuracy: {acc:.4f}")

Epoch 1/5
269/269 ━━━━━━━━━━━━━━━━━━━━ 20s 53ms/step - accuracy: 0.7286 - loss: 0.5212 - val_accuracy: 0.8819 - val_loss: 0.2793
Epoch 2/5
269/269 ━━━━━━━━━━━━━━━━━━━━ 11s 41ms/step - accuracy: 0.9211 - loss: 0.2147 - val_accuracy: 0.9143 - val_loss: 0.2440
Epoch 3/5
269/269 ━━━━━━━━━━━━━━━━━━━━ 20s 40ms/step - accuracy: 0.9540 - loss: 0.1406 - val_accuracy: 0.9064 - val_loss: 0.2615
Epoch 4/5
269/269 ━━━━━━━━━━━━━━━━━━━━ 11s 41ms/step - accuracy: 0.9670 - loss: 0.1064 - val_accuracy: 0.8944 - val_loss: 0.3148
Epoch 5/5
269/269 ━━━━━━━━━━━━━━━━━━━━ 20s 39ms/step - accuracy: 0.9772 - loss: 0.0751 - val_accuracy: 0.8944 - val_loss: 0.3133
150/150 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.8870 - loss: 0.3450
Test Accuracy: 0.8929


Accuracy: 0.8929

-- Transformers --

In [ ]:
# %pip install transformers datasets tensorflow scikit-learn


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from transformers import BertTokenizer, TFBertForSequenceClassification
from transformers import InputExample, InputFeatures
import tensorflow as tf

# 1. Prepare your data
X = balanced_data['text'].tolist()
y = balanced_data['lashon_hara_guess'].tolist()

# 2. Split
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=0.2, random_state=42)

# 3. Initialize tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

# 4. Convert to InputExamples
def convert_data(texts, labels):
    return [InputExample(guid=str(i), text_a=txt, text_b=None, label=str(lbl)) for i, (txt, lbl) in enumerate(zip(texts, labels))]

train_examples = convert_data(X_train, y_train)
test_examples = convert_data(X_test, y_test)

# 5. Convert InputExamples to InputFeatures
def convert_examples_to_tf_dataset(examples, tokenizer, max_length=128):
    features = []

    for e in examples:
        inputs = tokenizer.encode_plus(
            e.text_a,
            add_special_tokens=True,
            max_length=max_length,
            padding='max_length',
            truncation=True,
            return_token_type_ids=False,
            return_attention_mask=True,
        )
        features.append(InputFeatures(
            input_ids=inputs['input_ids'],
            attention_mask=inputs['attention_mask'],
            label=int(e.label)
        ))

    def gen():
        for f in features:
            yield ({
                'input_ids': f.input_ids,
                'attention_mask': f.attention_mask
            }, f.label)

    return tf.data.Dataset.from_generator(
        gen,
        output_types=({'input_ids': tf.int32, 'attention_mask': tf.int32}, tf.int32),
        output_shapes=({'input_ids': (128,), 'attention_mask': (128,)}, ())
    )

train_dataset = convert_examples_to_tf_dataset(train_examples, tokenizer).shuffle(1000).batch(32)
test_dataset = convert_examples_to_tf_dataset(test_examples, tokenizer).batch(32)

# 6. Load model
model = TFBertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)

# 7. Train
optimizer = tf.keras.optimizers.Adam(learning_rate=2e-5)
model.compile(optimizer=optimizer, loss=model.compute_loss, metrics=['accuracy'])

model.fit(train_dataset, epochs=3, validation_data=test_dataset)

# 8. Evaluate
loss, accuracy = model.evaluate(test_dataset)
print(f"Test accuracy: {accuracy:.4f}")


C:\Users\shmue\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ValueError: Your currently installed version of Keras is Keras 3, but this is not yet supported in Transformers. Please install the backwards-compatible tf-keras package with `pip install tf-keras`.

: 

In [ ]:
def predict_lashon_hara(text):
    inputs = tokenizer(text, return_tensors="tf", padding=True, truncation=True, max_length=128)
    logits = model(inputs)[0]
    predicted_class = tf.argmax(logits, axis=1).numpy()[0]
    return "Lashon Hara" if predicted_class == 1 else "Not Lashon Hara"

# Example:
predict_lashon_hara("@josh you're the dumbest person on this app")
